In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [14]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [15]:
# clone YOLOv3 implemementation
!git clone https://github.com/Lornatang/YOLOv3-PyTorch.git

fatal: destination path 'YOLOv3-PyTorch' already exists and is not an empty directory.


In [16]:
!ls YOLOv3-PyTorch

configs  figure   model_configs  requirements.txt  scripts   tools
data	 LICENSE  README.md	 results	   setup.py  yolov3_pytorch


In [20]:
# install YOLOv3
!ln -sf YOLOv3-PyTorch/yolov3_pytorch yolov3_pytorch
!ln -sf YOLOv3-PyTorch/tools tools
!ln -sf YOLOv3-PyTorch/configs configs
!ln -sf YOLOv3-PyTorch/model_configs model_configs

In [22]:
!pip install thop

In [23]:
# take images, e.g. using https://imageonline.io/take-photo/

In [24]:
# annotate images, using https://www.makesense.ai/ add label / select ROI + label / action / export / yolo

In [28]:
# download train and test images with annotations
!wget http://www.agentspace.org/download/watch-annotated.zip
!unzip -o watch-annotated.zip
!rm watch-annotated.zip

--2025-11-23 08:36:22--  http://www.agentspace.org/download/watch-annotated.zip
Resolving www.agentspace.org (www.agentspace.org)... 62.168.101.9
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.agentspace.org/download/watch-annotated.zip [following]
--2025-11-23 08:36:22--  https://www.agentspace.org/download/watch-annotated.zip
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1576196 (1.5M) [application/zip]
Saving to: ‘watch-annotated.zip’

watch-annotated.zip 100%[===================>]   1.50M  1.54MB/s    in 1.0s    

2025-11-23 08:36:24 (1.54 MB/s) - ‘watch-annotated.zip’ saved [1576196/1576196]

Archive:  watch-annotated.zip
 extracting: data/custom/custom.names  
  inflating: data/custom/images/test/000011.jpg  
  inflating: data/custom/images/test/000012.jpg  
  inflat

In [26]:
# download pretrained model
!wget -O YOLOv3_Tiny-VOC-20231107.pth.tar https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF

--2025-11-23 08:35:14--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF
Resolving drive.google.com (drive.google.com)... 142.250.101.113, 142.250.101.139, 142.250.101.138, ...
Connecting to drive.google.com (drive.google.com)|142.250.101.113|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/ [following]
--2025-11-23 08:35:14--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/
Reusing existing connection to drive.google.com:443.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/edit [following]
--2025-11-23 08:35:14--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/edit
Reusing existing connection to drive.google.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘YOLOv3_Tiny-VOC-20231107.pth.

In [27]:
!ls -lR model_configs/

model_configs/:
total 8
drwxr-xr-x 3 root root 4096 Nov 23 08:27 COCO-Detection
drwxr-xr-x 3 root root 4096 Nov 23 08:27 VOC-Detection

model_configs/COCO-Detection:
total 36
drwxr-xr-x 2 root root 4096 Nov 23 08:27 exp
-rw-r--r-- 1 root root 9450 Nov 23 08:27 yolov3.cfg
-rw-r--r-- 1 root root 9741 Nov 23 08:27 yolov3_spp.cfg
-rw-r--r-- 1 root root 2196 Nov 23 08:27 yolov3_tiny.cfg
-rw-r--r-- 1 root root 2097 Nov 23 08:27 yolov3_tiny_prn.cfg

model_configs/COCO-Detection/exp:
total 32
-rw-r--r-- 1 root root 9438 Nov 23 08:27 yolov3.cfg
-rw-r--r-- 1 root root 9729 Nov 23 08:27 yolov3_spp.cfg
-rw-r--r-- 1 root root 2190 Nov 23 08:27 yolov3_tiny.cfg
-rw-r--r-- 1 root root 2091 Nov 23 08:27 yolov3_tiny_prn.cfg

model_configs/VOC-Detection:
total 64
drwxr-xr-x 2 root root 4096 Nov 23 08:27 exp
-rw-r--r-- 1 root root 3892 Nov 23 08:27 yolov3_alexnet.cfg
-rw-r--r-- 1 root root 9447 Nov 23 08:27 yolov3.cfg
-rw-r--r-- 1 root root 4734 Nov 23 08:27 yolov3_mobilenetv1.cfg
-rw-r--r-- 1 root root 7

In [36]:
!cp -v model_configs/COCO-Detection/yolov3_tiny.cfg yolov3_tiny.cfg

'model_configs/COCO-Detection/yolov3_tiny.cfg' -> 'yolov3_tiny.cfg'


In [37]:
# edit yolov3_tiny.cfg and set the class number to 1
# change classes in each YOLO layer
# change filters to (5 + classes) * num_masked_anchors in the convolutional layer before the YOLO layer
# (e.g. we have 6 anchors with mask 0,1,2 i.e. we have used 3 anchors)
# (6*3=18)
!sed -i -e 's/^\s*filters\s*=\s*255/filters=18/' -e 's/^\s*classes\s*=\s*80/classes=1/' yolov3_tiny.cfg